In [ ]:
import argparse
import torch
import numpy as np
import random
from peft import (
    LoraConfig,
    get_peft_model
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
import sys
import json
import transformers
import warnings
from datasets import load_dataset
from predict_module import sft_dataloader
import re

from utils.prompts import PREDICT_INSTRUCTION
from utils.fewshots import PREDICT_EXAMPLES
# from torchdiffeq import odeint  # Cần cài đặt: pip install torchdiffeq
# import torch.nn as nn
# import torch.nn.functional as F


# Thiết lập seed
fix_seed = 100
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)


# Cấu hình tham số cho huấn luyện
args = argparse.Namespace(
    wandb=False,  # Tắt logging với Weights & Biases
    data_path="./data/DeepSeekLLM_top1_stock_merge_sample.json",  # Đường dẫn file dữ liệu tesst/KLTN/sep-main/data/DeepSeek_top1_merge_sample.json
    output_path="./saved_models/lora-DeepSeek-R1-Distill-Qwen",  # Thư mục lưu mô hình LoRA
    model_path="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình DeepSeek
    eval_steps=200,  # Số bước đánh giá
    save_steps=200,  # Số bước lưu checkpoint
    resume_from_supervised_checkpoint=None,  # Không resume từ checkpoint
    ignore_data_skip="False",  # Không bỏ qua dữ liệu khi resume
    num_reflect_trials=2,  # Số lần thử phản ánh
    datasets_dir="./datasets/",  # Thư mục datasets 
    datasets_for_train_flow="datasets_for_train_flow/data_policy_basic.json",  # Thư mục datasets 
    local_rank=0,  # Rank cục bộ cho DDP
    resume_from_reward_checkpoint=False,  # Không resume từ reward checkpoint
    deepspeed=None,  # Không dùng DeepSpeed
    per_device_train_batch_size=4,  # Batch size huấn luyện trên mỗi GPU
    per_device_eval_batch_size=4,  # Batch size đánh giá trên mỗi GPU
    reward_gradient_accumulation_steps=8,  # Số bước tích lũy gradient cho reward
    reward_learning_rate=3e-5,  # Learning rate cho reward
    weight_decay=0.001,  # Trọng số giảm dần
    reward_base_model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình reward
    bf16=False,  # Sử dụng fp16 thay vì bf16
    num_train_epochs=2,  # Số epoch huấn luyện
    train_subset=100000,  # Số mẫu huấn luyện
    eval_subset=50000,  # Số mẫu đánh giá
    gradient_checkpointing=True,  # Bật gradient checkpointing để tiết kiệm VRAM
    optim="adamw_torch",  # Optimizer AdamW từ PyTorch
    lr_scheduler_type="cosine",  # Lịch trình learning rate kiểu cosine
    reward_adapter="./saved_models/reward_model_deepseek-r1-distill-qwen",  # Adapter reward
    rl_base_model="./saved_models/lora-DeepSeek-R1-Distill-Qwen-adapter-merged",  # Mô hình RL
    # rl_base_model= "./saved_models/lora-DeepSeek-R1-Distill-Qwen",
    tokenizer_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Tokenizer
    reward_model_name="./saved_models/reward_model_deepseek-r1-distill-qwen-adapter-merged",  # Mô hình reward merged
    log_with=None,  # Không dùng logging cụ thể
    rl_learning_rate=2e-5,  # Learning rate cho RL
    output_max_length=128,  # Độ dài đầu ra tối đa
    mini_batch_size=4,  # Kích thước mini-batch
    batch_size=128,  # Kích thước batch tổng
    ppo_epochs=2,  # Số epoch cho PPO
    rl_gradient_accumulation_steps=32,  # Số bước tích lũy gradient cho RL
    adafactor=False,  # Không dùng Adafactor
    early_stopping=True,  # Bật early stopping
    target_kl=0.1,  # KL target cho RL
    reward_baseline=0,  # Baseline cho reward
    batched_gen=True,  # Tạo batch
    save_freq=100,  # Tần suất lưu
    output_dir="./saved_models/tuning_deepseek_r1_distill_qwen_checkpoints/",  # Thư mục lưu checkpoint
    seed=0,  # Seed cho RL
    num_shots=4,  # Số shots cho few-shot
    save_dir="results/"  # Thư mục lưu kết quả
)

def split_completion(completion_text):
    completion_text = completion_text.replace("*", "")
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)


        # Dò tìm nhãn Positive/Negative trong cả phần text
        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            target = match.group(1).capitalize()
        else:
            target = 'Mixed'

        # Làm sạch phần giải thích
        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        target = 'Mixed'
        explain = ""
        
    return target, explain

LLM_GUIDANCE_MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"


In [ ]:
# from datasets import load_dataset, DatasetDict
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     TrainingArguments,
#     Trainer,
# )
# # from trl import SFTTrainer
# import torch
# from peft import LoraConfig, get_peft_model, set_peft_model_state_dict


# def supervised_finetune(args):
#     # --- Các hằng số huấn luyện ---
#     MICRO_BATCH_SIZE = args.per_device_train_batch_size  # Batch size mỗi GPU
#     BATCH_SIZE = args.batch_size  # Batch size tổng
#     MAX_STEPS = None  # Số bước tối đa, tính động
#     GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE  # Bước tích lũy gradient
#     EPOCHS = args.num_train_epochs  # Số epoch
#     LEARNING_RATE = 3e-4  # Tốc độ học
#     CUTOFF_LEN = 256  # Độ dài chuỗi tối đa
#     LORA_R = 16  # Rank LoRA
#     LORA_ALPHA = 32  # Hệ số scale LoRA
#     LORA_DROPOUT = 0.05  # Dropout LoRA
#     VAL_PCT = 0.1  # Tỷ lệ validation
#     TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # Layer áp dụng LoRA
#     DATA_PATH = args.data_path  # Đường dẫn dữ liệu
#     OUTPUT_DIR = args.output_path  # Thư mục lưu mô hình
#     world_size = int(os.environ.get("WORLD_SIZE", 1))  # Số GPU (DDP)


#     # --- Xử lý DDP ---
#     ddp = world_size != 1  # Kiểm tra đa GPU
#     if ddp:
#         torch.cuda.set_device(int(os.environ.get("LOCAL_RANK", 0)))  # Gán GPU
#         GRADIENT_ACCUMULATION_STEPS = GRADIENT_ACCUMULATION_STEPS // world_size  # Chia tích lũy gradient

#     #==============================================================================================================================
#     print(f"Đang tải mô hình từ: {args.model_path}")  # In đường dẫn mô hình

#     # Step 3: Load model and tokenizer
#     tokenizer = AutoTokenizer.from_pretrained(
#         args.model_path,
#         add_eos_token=True,  # Thêm token kết thúc
#         local_files_only=args.offline if hasattr(args, 'offline') else False  # Chế độ offline
#         )

#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token  # Gán pad_token

#     model = AutoModelForCausalLM.from_pretrained(
#         args.model_path, 
#         torch_dtype=torch.float16, 
#         device_map="auto")

#     # --- Tải dữ liệu ---
#     dataset = load_dataset("json", data_files=DATA_PATH)  # Tải JSON dataset
#     # dataset = DatasetDict({"train": dataset["train"].select([0, 1])})


#     val_set_size = int(VAL_PCT * len(dataset["train"]))  # Tính validation size
#     print(dataset)  # In thông tin dataset


#     # --- Tải dữ liệu huấn luyện và đánh giá ---
#     dataloader = sft_dataloader.SFTDataLoader(dataset, CUTOFF_LEN, val_set_size, tokenizer)  # Định dạng và tokenize
#     train_data, val_data = dataloader.load_data()  # Chia train/validation
#     train_data, val_data

#     # Step 4: Configure LoRA
#     peft_config = LoraConfig(
#         r=LORA_R,  # Rank LoRA
#         lora_alpha=LORA_ALPHA,  # Scale LoRA
#         target_modules=TARGET_MODULES,  # Layer LoRA
#         lora_dropout=LORA_DROPOUT,  # Dropout
#         bias="none",  # Không bias
#         task_type="CAUSAL_LM"  # Tác vụ ngôn ngữ
#     )


#     model = get_peft_model(model, peft_config)


#     # --- Tính max_steps ---
#     now_max_steps = max((len(dataset["train"]) - val_set_size) // BATCH_SIZE * EPOCHS, EPOCHS)  # Số bước tối đa


#     # --- Xử lý checkpoint ---
#     if args.resume_from_supervised_checkpoint:  # Nếu có checkpoint
#         checkpoint_name = os.path.join(args.resume_from_supervised_checkpoint, "pytorch_model.bin")  # Đường dẫn checkpoint
#         if not os.path.exists(checkpoint_name):
#             pytorch_bin_path = checkpoint_name
#             checkpoint_name = os.path.join(args.resume_from_supervised_checkpoint, "adapter_model.bin")  # Kiểm tra file khác
#             if os.path.exists(checkpoint_name):
#                 os.rename(checkpoint_name, pytorch_bin_path)  # Đổi tên
#                 warnings.warn("Đã đổi tên 'adapter_model.bin' thành 'pytorch_model.bin'")
#             else:
#                 args.resume_from_supervised_checkpoint = None  # Bỏ resume
#         if os.path.exists(checkpoint_name):
#             print(f"Tiếp tục từ: {checkpoint_name}")
#             adapters_weights = torch.load(checkpoint_name)  # Tải LoRA
#             model = set_peft_model_state_dict(model, adapters_weights)  # Áp dụng
#         else:
#             print(f"Không tìm thấy: {checkpoint_name}")
#         train_args_path = os.path.join(args.resume_from_supervised_checkpoint, "trainer_state.json")  # File trạng thái
#         if os.path.exists(train_args_path):
#             base_train_args = json.load(open(train_args_path, 'r'))
#             base_max_steps = base_train_args["max_steps"]  # Số bước cũ
#             resume_scale = base_max_steps / now_max_steps
#             if base_max_steps > now_max_steps:
#                 warnings.warn(f"Thay epoch {EPOCHS} bằng {base_max_steps}")
#                 EPOCHS = None
#                 MAX_STEPS = base_max_steps
#             else:
#                 MAX_STEPS = now_max_steps
#     else:
#         MAX_STEPS = now_max_steps


#     # Step 5: Define training arguments
#     training_args = TrainingArguments(
#         output_dir=OUTPUT_DIR,  # Directory to save results
#         per_device_train_batch_size=MICRO_BATCH_SIZE,  # Batch size GPU
#         gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,  # Tích lũy gradient
#         warmup_steps=100,  # Bước khởi động
#         num_train_epochs=EPOCHS if EPOCHS else 1,  # Epoch
#         max_steps=MAX_STEPS,  # Số bước tối đa
#         learning_rate=LEARNING_RATE,  # Tốc độ học
#         bf16=args.bf16,  # BF16
#         fp16=not args.bf16,  # FP16
#         logging_steps=20,  # Log mỗi 20 bước
#         eval_strategy="steps" if val_set_size > 0 else "no",  # Đánh giá
#         save_strategy="steps",  # Lưu checkpoint
#         eval_steps=args.eval_steps if val_set_size > 0 else None,  # Bước đánh giá
#         save_steps=args.save_steps,  # Bước lưu
#         save_total_limit=30,  # Số checkpoint tối đa
#         load_best_model_at_end=True if val_set_size > 0 else False,  # Tải mô hình tốt
#         ddp_find_unused_parameters=False if ddp else None,  # Tối ưu DDP
#         report_to="wandb" if args.wandb else [],  # Báo cáo WandB
#         optim=args.optim,  # Bộ tối ưu
#         lr_scheduler_type=args.lr_scheduler_type,  # Scheduler
#         remove_unused_columns=True,  # Xóa cột thừa
#         max_grad_norm=1.0,  # Giới hạn gradient
#     )


#     # Step 6: Initialize the trainer
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_data, 
#         eval_dataset=val_data,  # Small evaluation set
#         data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
#     )

#     # Step 7: Train the model
#     trainer.train(resume_from_checkpoint=args.resume_from_supervised_checkpoint)
    
#     model.save_pretrained(OUTPUT_DIR) 

# # --- Chạy huấn luyện ---

# supervised_finetune(args)

# from predict_module.merge_peft_adapter import merge_peft_adapter

# # --- Gộp adapter LoRA ---
# merge_peft_adapter(model_name=args.output_path, output_name=args.rl_base_model)

## SEPP SFT ONLY

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import pandas as pd
import re
import json

def split_completion(completion_text):
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)

        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            target = match.group(1).capitalize()
        else:
            target = 'Mixed'

        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        target = 'Mixed'
        explain = ""
        
    return target, explain

# Đường dẫn model và tokenizer
model_path = args.rl_base_model
tokenizer_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)

# Load dataset từ CSV tesst/KLTN/Data/summarized/OpenAILLM_top1_stock_data_test.csv
data_path = "../Data/summarized/OpenAILLM_top1_stock_data_test.csv"
test_ds = pd.read_csv(data_path)

# Chuẩn bị danh sách lưu kết quả
data_result = []

# Prompt template (bạn cần chắc chắn rằng PREDICT_INSTRUCTION và PREDICT_EXAMPLES đã được định nghĩa trước)

# Lặp qua từng mẫu dữ liệu
i = 1
for i, sample in test_ds.iterrows():
    print(f"ĐANG THỰC HIỆN MẪU i = {i}")
    i += 1

    ticker = sample["ticker"]
    summary = sample["summary"]
    label = sample["target"]

    prompt = PREDICT_INSTRUCTION.format(
        examples=PREDICT_EXAMPLES,
        ticker=ticker,
        summary=summary
    )

    # print("\n--- Prompt ---\n", prompt)

    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        top_p=0.95,
        top_k=0,
        temperature=0.6,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode và in ra response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n--- Response ---\n", response)

    predict, explain = split_completion(response)

    data_result.append({
        "user_input": prompt,
        "prediction_of_LLM": predict,
        "explain": explain,
        "Label": label
    })

# Lưu kết quả ra file CSV
df = pd.DataFrame(data_result)
df.to_csv("SFT_RESULTS_top1_stock.csv", index=False, encoding="utf-8-sig")

# Dọn dẹp bộ nhớ GPU
torch.cuda.empty_cache()
from sklearn.metrics import accuracy_score, matthews_corrcoef

# Giả sử df là DataFrame của bạn
y_pred = df["prediction_of_LLM"]
y_true = df["Label"]

# Tính accuracy
acc = accuracy_score(y_true, y_pred)

# Tính MCC
mcc = matthews_corrcoef(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"MCC: {mcc:.4f}")

## PPO WITHOUT FLOW MODEL

### GUIDANCE

In [ ]:
import json
import re

def convert_prompt_with_regex(text: str) -> str:
    pattern = re.compile(
        r"Given a list of facts and a set of technical indicators, estimate their overall impact on the price movement of (?P<ticker>\w+) stock\. "
        r"Give your response in this format:\n"
        r"\(1\) Price Movement, which should be either Positive or Negative\.\n"
        r"\(2\) Explanation, which should be in a single, short paragraph\.\n"
        r"Here are some examples:",
        re.MULTILINE
    ) 
        
    replacement = (
        "You are given a list of chronological facts spanning multiple days about \\g<ticker> and a set of technical indicators. "
        "Your task is to assess the overall impact of these facts and technical indicators on the stock price and provide:\n\n"
        "(1) One price Movement:: either Positive or Negative — no other responses are allowed.\n"
        "(2) One explanation: a short, single-paragraph justification summarizing the main reasons behind your prediction.\n\n"
        "Do not output anything other than these two parts.\n"
        "Here is a examples, the contents of the example are not used to deduce the answer:"
    )

    return pattern.sub(replacement, text)

def split_completion(completion_text):
    completion_text = completion_text.replace("*", "")
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)


        # Dò tìm nhãn Positive/Negative trong cả phần text
        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            predict = match.group(1).capitalize()
        else:
            predict = 'Mixed'

        # Làm sạch phần giải thích
        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        predict = 'Mixed'
        explain = ""
        
    return predict, explain

with open("datasets/DeepSeekLLM_top1_stock_comparison_data.json", "r", encoding="utf-8") as f:  # tesst/KLTN/sep-main-update/datasets/FULL_DeepSeekLLM_top1_stock_technical_indicator_comparison_data.json
    data = json.load(f)

data_policy =[]
data_comparision =[]

for i, example in enumerate(data):
    user_input = example["user_input"]
    user_input = convert_prompt_with_regex(user_input)

    completion_a = example["completion_a"]
    predict_a, explain_a = split_completion(completion_a)
    
    completion_b = example["completion_b"]
    predict_b, explain_b = split_completion(completion_b)

    label = predict_b
    
    if(label in ['Positive','Negative','positive','negative']):
        if(predict_a in ['Positive','Negative','positive','negative']):
            data_policy.append({"user_input": user_input, "prediction_of_LLM": predict_a, "explain": explain_a, "Label": label})
        data_policy.append({"user_input": user_input, "prediction_of_LLM": predict_b, "explain": explain_b, "Label": label})

        data_comparision.append({"user_input": user_input, "completion_a": completion_a, "completion_b": completion_b, "Label": label})

    
print(len(data_policy))

# Lưu vào file JSON 
with open(args.datasets_for_train_flow, "w", encoding="utf-8") as f:
    json.dump(data_policy, f, ensure_ascii=False, indent=2)

# Lưu vào file JSON 
with open("datasets/DeepSeekLLM_top1_stock_comparison_data.json", "w", encoding="utf-8") as f:
    json.dump(data_comparision, f, ensure_ascii=False, indent=2)


451


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer
from torchdiffeq import odeint  # Cần cài đặt: pip install torchdiffeq
import numpy as np
from torch.nn.utils.rnn import pad_sequence


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LLM_GUIDANCE_MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

MAX_SEQ_LENGTH_GUIDANCE = 256
LEARNING_RATE_FLOW_MODEL = 2e-4

# Lớp Guidance LLM hướng đối tượng
class GuidanceLLM:
    """
        Khởi tạo Guidance LLM.
        Input: model_name (str): tên mô hình LLM (mặc định là "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
        Output: Không có giá trị trả về trực tiếp, chỉ khởi tạo các thuộc tính.
    """
    def __init__(self, model_name=LLM_GUIDANCE_MODEL_NAME):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.hidden_dim = None
        self.load_model()

    def load_model(self):
        """
        Tải mô hình và tokenizer.
        Input: Không có input trực tiếp.
        Output: Không có giá trị trả về, chỉ in thông báo và khởi tạo model, tokenizer.
        """
        print(f"Loading Guidance LLM: {self.model_name}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.bfloat16,
                trust_remote_code=True
            ).to(DEVICE)
            
            for param in self.model.parameters():
                param.requires_grad = False
            self.model.eval()

            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                self.model.config.pad_token_id = self.model.config.eos_token_id

            self.hidden_dim = self.model.config.hidden_size
            print(f"Guidance LLM loaded on {DEVICE}. Hidden dim: {self.hidden_dim}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def get_decision_distribution(self, user_input_text, explanation_sentences_so_far, all_possible_actions):
        """
        Tính phân phối xác suất cho các hành động dựa trên ngữ cảnh và giải thích từng phần.
        Input:
            - user_input_text (str): ngữ cảnh/quan sát của tác nhân.
            - explanation_sentences_so_far (list): danh sách các câu giải thích đã sinh ra (từng phần).
            - all_possible_actions (list): danh sách các hành động/quyết định có thể.
        Output:
            - probability_distribution (tensor): tensor xác suất (softmax) cho từng hành động/quyết định.
        """
        current_explanation = " ".join(explanation_sentences_so_far)
        action_logits_list = []

        with torch.no_grad():
            for action_text in all_possible_actions:
                prompt = (
                    f"Given the following information about a company: '{user_input_text}'.\n"
                    f"Based on the reasoning: '{current_explanation}'.\n"
                    f"Predict the price impact: {action_text}."
                )
                inputs = self.tokenizer(
                    prompt,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_SEQ_LENGTH_GUIDANCE
                ).to(DEVICE)

                try:
                    outputs = self.model(**inputs)
                    logits = outputs.logits[:, -1, :]  # Logits token cuối
                    action_tokens = self.tokenizer(action_text, add_special_tokens=False).input_ids
                    if not action_tokens:
                        action_logits_list.append(torch.tensor(-10.0).to(DEVICE))
                        continue

                    # Giả lập: Trung bình logits (cần fine-tune thực tế)
                    logit_score = logits.mean()
                    action_logits_list.append(logit_score)
                except Exception as e:
                    print(f"Error processing action '{action_text}': {e}")
                    action_logits_list.append(torch.tensor(-10.0).to(DEVICE))

        if not action_logits_list or all(l.item() == -10.0 for l in action_logits_list):
            return torch.ones(len(all_possible_actions), device=DEVICE) / len(all_possible_actions)

        logits_tensor = torch.stack(action_logits_list)
        probability_distribution = torch.softmax(logits_tensor, dim=0)
        return probability_distribution

    def get_last_layer_hidden_states(self, user_input_text, explanation_sentences_so_far, all_possible_actions_text_for_prompt):
        
        """
        Lấy hidden states từ tầng cuối của LLM.
        Input:
            - user_input_text (str): ngữ cảnh/quan sát của tác nhân.
            - explanation_sentences_so_far (list): danh sách các câu giải thích đã sinh ra (từng phần).
            - all_possible_actions_text_for_prompt (str): chuỗi mô tả các hành động/quyết định để ghép vào prompt.
        Output:
            - hidden_states (tensor): tensor hidden states từ tầng cuối của LLM, shape [batch_size, seq_len, hidden_dim].
        """
        current_explanation = " ".join(explanation_sentences_so_far)
        prompt = (
            f"Given the following information about a company: '{user_input_text}'.\n"
            f"Based on the reasoning: '{current_explanation}'.\n"
            f"Consider the possible outcomes: {all_possible_actions_text_for_prompt}."
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH_GUIDANCE
        ).to(DEVICE)

        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1]  # [batch_size, seq_len, hidden_dim]
    

    def collect_positive_samples(self, sample_dataset, possible_actions):
        """
        Thu thập mẫu dương: Các mẫu mà Guidance LLM dự đoán đúng nhãn thực tế (Label) khi nhận input là user_input và từng phần của explain.
        Input:
            - sample_dataset (list): danh sách các dict, mỗi dict có keys: user_input, prediction_of_LLM, explain, Label
            - possible_actions (list): danh sách các hành động/quyết định có thể, ví dụ: ["negative", "positive"]
        Output:
            - positive_samples (list): danh sách các tuple (z1, context, explanation, hidden_states)
              trong đó:
                - z1 (tensor): phân phối mục tiêu từ Guidance LLM (khi dự đoán đúng Label thực tế)
                - context (str): user_input – thông tin fact
                - explanation (list): danh sách các câu giải thích (từng phần)
                - hidden_states (tensor): hidden states từ tầng cuối của Guidance LLM
        """
        possible_actions = [text.lower() for text in possible_actions]
        positive_samples = []
        for sample in sample_dataset:
            context = sample['user_input']
            true_action = sample['Label'].lower()  # Lấy nhãn thực tế từ 'Label'
            explanation = sample['explain']
            # Tách giải thích thành các câu
            sentences = [s.strip() for s in explanation.split('.') if s.strip()]
            print("len(sentences) = ", len(sentences))

            for k in range(1, len(sentences) + 1):
                # print("asdasd", k, "sadsad")
                partial_explanation = sentences[:k]
                prob_dist = self.get_decision_distribution(context, partial_explanation, possible_actions)
                hidden_states = self.get_last_layer_hidden_states(context, partial_explanation, ", ".join(possible_actions))
                # Kiểm tra xem nhãn thực tế có khả năng cao nhất không
                if torch.argmax(prob_dist).item() == possible_actions.index(true_action):
                    positive_samples.append((
                        prob_dist.detach(),
                        context,
                        partial_explanation,
                        hidden_states
                    ))
        return positive_samples

# Mô hình Rectified Flow
class RectifiedFlowModel(nn.Module):
    """
        Khởi tạo mô hình Rectified Flow.
        Input:
            - guidance_llm_instance: instance của GuidanceLLM.
            - num_actions (int): số lượng hành động/quyết định.
            - flow_embed_dim (int): kích thước embedding cho flow tokens.
            - projector_hidden_dim (int): kích thước hidden layer của projector MLP.
            - projector_layers (int): số layer projector MLP.
            - num_attention_heads (int): số attention heads cho cross-attention.
            - dropout_rate (float): dropout rate cho attention.
        Output: Không có giá trị trả về, chỉ khởi tạo mô hình.
    """

    def __init__(
        self,
        guidance_llm_instance,  # Instance của GuidanceLLM
        num_actions,
        flow_embed_dim=256,
        projector_hidden_dim=256,
        projector_layers=4,
        num_attention_heads=4,
        dropout_rate=0.1
    ):
        super().__init__()
        self.guidance_llm_instance = guidance_llm_instance
        self.num_actions = num_actions
        self.flow_embed_dim = flow_embed_dim
        self.projector_hidden_dim = projector_hidden_dim
        self.projector_layers = projector_layers
        self.num_attention_heads = num_attention_heads

        # Kích thước hidden state từ Guidance LLM
        self.hidden_dim_guidance_llm = self.guidance_llm_instance.hidden_dim

        # Embedding cho z_t và t
        self.zt_embed_mlp = nn.Sequential(
            nn.Linear(num_actions, flow_embed_dim),
            nn.ReLU(),
            nn.LayerNorm(flow_embed_dim)
        )
        self.time_embed_mlp = nn.Sequential(
            nn.Linear(1, flow_embed_dim),
            nn.ReLU(),
            nn.LayerNorm(flow_embed_dim)
        )

        # Truy cập W_Q, W_K, W_V từ Guidance LLM
        try:
            # Qwen-1.5B: transformer.h[-1].attn.q_proj/k_proj/v_proj
            last_guidance_layer_attn = self.guidance_llm_instance.model.transformer.h[-1].attn
            self.guidance_wq = last_guidance_layer_attn.q_proj
            self.guidance_wk = last_guidance_layer_attn.k_proj
            self.guidance_wv = last_guidance_layer_attn.v_proj
            print("Successfully accessed Q, K, V projection weights from Guidance LLM.")

            # Đóng băng trọng số
            for param in [self.guidance_wq.parameters(), self.guidance_wk.parameters(), self.guidance_wv.parameters()]:
                for p in param:
                    p.requires_grad = False
            self.using_borrowed_weights = True
        except AttributeError as e:
            print(f"Warning: Could not access Q, K, V weights ({e}). Using new Linear layers.")
            self.flow_qkv_proj = nn.Linear(flow_embed_dim, self.hidden_dim_guidance_llm)
            self.guidance_kv_proj = nn.Linear(self.hidden_dim_guidance_llm, self.hidden_dim_guidance_llm)
            self.using_borrowed_weights = False

        # Cross-attention
        self.attention_layer = nn.MultiheadAttention(
            embed_dim=self.hidden_dim_guidance_llm,
            num_heads=num_attention_heads,
            dropout=dropout_rate,
            batch_first=True
        )
        self.attn_output_norm = nn.LayerNorm(self.hidden_dim_guidance_llm)
        self.attn_output_projection = nn.Linear(self.hidden_dim_guidance_llm, flow_embed_dim)

        # Projector MLP
        projector_modules = []
        current_dim = flow_embed_dim
        for i in range(projector_layers):
            output_dim = projector_hidden_dim if i < projector_layers - 1 else num_actions
            projector_modules.append(nn.Linear(current_dim, output_dim))
            if i < projector_layers - 1:
                projector_modules.extend([
                    nn.ReLU(),
                    nn.LayerNorm(output_dim),
                    nn.Dropout(dropout_rate)
                ])
                current_dim = output_dim
        self.projector_mlp = nn.Sequential(*projector_modules)

        self.optimizer = optim.AdamW(self.parameters(), lr=LEARNING_RATE_FLOW_MODEL)

    def _project_for_attention(self, tensor, projection_type):
        """
        Áp dụng phép chiếu Q, K, V cho cross-attention.
        Input:
            - tensor (tensor): tensor đầu vào.
            - projection_type (str): 'q', 'k', hoặc 'v'.
        Output:
            - tensor (tensor): tensor sau khi chiếu theo Q, K hoặc V.
        """

        if self.using_borrowed_weights:
            if projection_type == 'q': return self.guidance_wq(tensor)
            elif projection_type == 'k': return self.guidance_wk(tensor)
            elif projection_type == 'v': return self.guidance_wv(tensor)
        else:
            if projection_type == 'q': return self.flow_qkv_proj(tensor)
            elif projection_type in ['k', 'v']: return self.guidance_kv_proj(tensor)
        raise ValueError(f"Unknown projection_type: {projection_type}")

    def forward(self, z_t, time_t, guidance_llm_last_hidden_states):
        """
        Forward pass của Rectified Flow Model.
        Input:
            - z_t (tensor): vector noise (flow state), shape [batch_size, num_actions].
            - time_t (tensor): thời gian flow, shape [batch_size, 1].
            - guidance_llm_last_hidden_states (tensor): hidden states từ LLM, shape [batch_size, seq_len, hidden_dim].
        Output:
            - vector_field (tensor): tensor vector field dự đoán, shape [batch_size, num_actions].
        """
        z_t = z_t.to(torch.bfloat16)
        time_t = time_t.to(torch.bfloat16)
        guidance_llm_last_hidden_states = guidance_llm_last_hidden_states.to(torch.bfloat16)


        batch_size = z_t.shape[0]

        # Embedding z_t và t
        h_emb_zt = self.zt_embed_mlp(z_t)  # [batch_size, flow_embed_dim]
        h_emb_t = self.time_embed_mlp(time_t)  # [batch_size, flow_embed_dim]
        flow_tokens_emb = (h_emb_zt + h_emb_t).unsqueeze(1)  # [batch_size, 1, flow_embed_dim]

        # Cross-attention
        query = self._project_for_attention(flow_tokens_emb, 'q')  # [batch_size, 1, hidden_dim_guidance_llm]
        key = self._project_for_attention(guidance_llm_last_hidden_states, 'k')  # [batch_size, seq_len, hidden_dim_guidance_llm]
        value = self._project_for_attention(guidance_llm_last_hidden_states, 'v')  # [batch_size, seq_len, hidden_dim_guidance_llm]

        attn_output, _ = self.attention_layer(query, key, value, need_weights=False)
        attn_output_normalized = self.attn_output_norm(attn_output)
        h_attn_zt = self.attn_output_projection(attn_output_normalized.squeeze(1))  # [batch_size, flow_embed_dim]

        # Projector
        vector_field = self.projector_mlp(h_attn_zt)  # [batch_size, num_actions]
        return vector_field

    def train_step(self, z0_batch, z1_batch, time_batch, guidance_llm_hidden_states_batch):
        """
        Thực hiện một bước huấn luyện cho flow model.
        Input:
            - z0_batch (tensor): batch vector noise ban đầu.
            - z1_batch (tensor): batch vector mục tiêu (phân phối mẫu dương).
            - time_batch (tensor): batch thời gian flow.
            - guidance_llm_hidden_states_batch (tensor): batch hidden states từ LLM.
        Output:
            - loss.item() (float): giá trị loss sau một bước huấn luyện.
        """

        self.optimizer.zero_grad()
        z_t_batch = time_batch * z1_batch + (1.0 - time_batch) * z0_batch
        target_vector_field = z1_batch - z0_batch
        predicted_vector_field = self.forward(z_t_batch, time_batch, guidance_llm_hidden_states_batch)
        loss = nn.MSELoss()(predicted_vector_field, target_vector_field)
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    def train_on_positive_samples(self, positive_samples, epochs=10, batch_size=16, print_every=10):
        """
        Huấn luyện FlowModel trên tập mẫu dương.
        Input:
            - positive_samples (list): danh sách các tuple (z1, context, explanation, hidden_states)
              trong đó:
                - z1 (tensor): phân phối mục tiêu từ Guidance LLM (khi dự đoán đúng action thực tế)
                - context (str): ngữ cảnh/quan sát
                - explanation (list): danh sách các câu giải thích (từng phần)
                - hidden_states (tensor): hidden states từ tầng cuối của Guidance LLM
            - epochs (int): số epoch huấn luyện
            - batch_size (int): kích thước batch
            - print_every (int): in loss sau mỗi print_every epoch
        Output: Không có giá trị trả về trực tiếp, chỉ thực hiện huấn luyện và in loss.
        """
        for epoch in range(epochs):
            # Trộn mẫu
            np.random.shuffle(positive_samples)
            total_loss = 0.0
            for i in range(0, len(positive_samples), batch_size):
                batch = positive_samples[i:i+batch_size]
                # Chuẩn bị batch
                z1_batch = torch.stack([sample[0] for sample in batch]).to(DEVICE)
                # hidden_states_batch = torch.cat([sample[3] for sample in batch], dim=0).to(DEVICE)
                hidden_states_list = [sample[3].squeeze(0) for sample in batch]  # [seq_len, hidden_dim]
                hidden_states_padded = pad_sequence(hidden_states_list, batch_first=True)  # [batch_size, max_seq_len, hidden_dim]
                hidden_states_batch = hidden_states_padded.to(DEVICE)
                
                z0_batch = torch.randn_like(z1_batch).to(DEVICE)
                time_batch = torch.rand(len(batch), 1).to(DEVICE)
                # Huấn luyện
                loss = self.train_step(z0_batch, z1_batch, time_batch, hidden_states_batch)
                total_loss += loss * len(batch)
            avg_loss = total_loss / len(positive_samples)
            if epoch % print_every == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch}, Loss: {avg_loss:.4f}")

    @torch.no_grad()
    def solve_ode_generate_distribution(self, z0, guidance_llm_last_hidden_states, num_steps=10, return_logits=False):
        """
        Giải ODE để sinh ra phân phối dự đoán.
        Input:
            - z0 (tensor): vector noise ban đầu, shape [batch_size, num_actions].
            - guidance_llm_last_hidden_states (tensor): hidden states từ LLM, shape [batch_size, seq_len, hidden_dim].
            - num_steps (int): số bước giải ODE.
            - return_logits (bool): trả về logits hay phân phối softmax.
        Output:
            - output (tensor): phân phối dự đoán sau khi giải ODE, shape [batch_size, num_actions] (softmax hoặc logits).
        """
        
        self.eval()
        if z0.ndim == 1: z0 = z0.unsqueeze(0)
        if guidance_llm_last_hidden_states.ndim == 2: guidance_llm_last_hidden_states = guidance_llm_last_hidden_states.unsqueeze(0)

        z0 = z0.to(torch.bfloat16)
        guidance_llm_last_hidden_states = guidance_llm_last_hidden_states.to(torch.bfloat16)
        
        batch_size = z0.shape[0]

        def ode_func(t, z):
            time_t = torch.full((batch_size, 1), t, device=DEVICE)
            return self.forward(z, time_t, guidance_llm_last_hidden_states)

        t = torch.linspace(0, 1, num_steps, device=DEVICE)
        z_t = odeint(ode_func, z0, t, method='rk4')  # Runge-Kutta
        z1_hat = z_t[-1]
        output = z1_hat.squeeze(0) if batch_size == 1 else z1_hat
        self.train()
        return output if return_logits else torch.softmax(output, dim=-1)
    

In [ ]:
import torch
def print_vram_usage():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(DEVICE) / 1024**3  # GB
        reserved = torch.cuda.memory_reserved(DEVICE) / 1024**3   # GB
        print(f"VRAM Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")


import torch
import torch.nn as nn
import numpy as np
import random
from datasets import load_dataset, concatenate_datasets
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    GenerationConfig, DataCollatorWithPadding
)
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from peft import LoraConfig
import os

# # Thiết lập seed
# fix_seed = 100
# random.seed(fix_seed)
# torch.manual_seed(fix_seed)
# np.random.seed(fix_seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
POSSIBLE_ACTIONS = ["Negative", "Positive"]
# Cấu hình mô hình Flow Matching và Guidance LLM (giả sử đã có)
# Dưới đây là khởi tạo giả định, bạn sửa lại theo code thực tế của bạn
# from rectified_flow_model import RectifiedFlowModel, GuidanceLLM  # Giả sử đã import class này từ file khác

def build_dataset(tokenizer, dataset_name, input_min_text_length=2, input_max_text_length=8):
    """
    Tạo dataset cho huấn luyện PPO.
    Input:
        - tokenizer: Tokenizer để mã hóa văn bản.
        - dataset_name: Tên hoặc đường dẫn dataset.
        - input_min_text_length, input_max_text_length: giữ nguyên
    Output:
        - Dataset đã được xử lý với các cột query, input_ids, label.
    """
    ds = load_dataset(dataset_name, split="train")
    # repeat_factor = 8 // len(ds) + 1
    # ds = concatenate_datasets([ds] * repeat_factor)
    # ds = ds.select(range(8))
    original_columns = ds.column_names

    def preprocess_function(examples):
        new_examples = {
            "query": [],
            "input_ids": [],
            "label": [],
        }
        for context, label in zip(examples["user_input"], examples["Label"]):
            query = "Given the context: '" + context + "'. Please analyze reasoning for the agent decision based on the context."
            tokenized_question = tokenizer(
                query,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )
            new_examples["query"].append(query)
            new_examples["input_ids"].append(tokenized_question["input_ids"].squeeze(0))
            new_examples["label"].append(label)
        return new_examples

    ds = ds.map(
        preprocess_function,
        batched=True,
        num_proc=1,
        remove_columns=original_columns,
    )
    ds.set_format(type="torch")
    return ds

def collator(data):
    """
    Collator để xử lý batch dữ liệu.
    Input: list[dict] với các key: query, input_ids, label
    Output: dict với các key: query, input_ids, label
    """
    return_dict = {
        "query": [d["query"] for d in data],
        "input_ids": [d["input_ids"].to(DEVICE) for d in data],  # Giữ danh sách tensor 1D
        "label": [d["label"] for d in data],
    }
    return return_dict
    return dict((key, [d[key] for d in data]) for key in data[0])
    # return {
    #     "query": [d["query"] for d in data],
    #     "input_ids": [d["input_ids"] for d in data],
    #     "label": [d["label"] for d in data],
    # }

def compute_reward(flow_model, guidance_llm, context, explanation, true_action, possible_actions):
    """
    Tính reward cho toàn bộ explanation dựa trên Flow Matching Model.
    Input:
        - flow_model: RectifiedFlowModel đã huấn luyện
        - guidance_llm: GuidanceLLM
        - context (str): ngữ cảnh/quan sát
        - explanation (str): giải thích đầy đủ
        - true_action (str): hành động đúng (Label)
        - possible_actions (list): danh sách hành động/quyết định có thể
    Output:
        - reward (float): reward cho explanation hiện tại 
    """
    
    hidden_states = guidance_llm.get_last_layer_hidden_states(
        context, explanation, ", ".join(possible_actions)
    )
    z0 = torch.randn(1, len(possible_actions)).to(DEVICE)
    p_hat_dist = flow_model.solve_ode_generate_distribution(z0, hidden_states, num_steps=5)
    true_idx = possible_actions.index(true_action)
    reward = p_hat_dist[true_idx].item()

    del hidden_states, z0, p_hat_dist
    torch.cuda.empty_cache()
        
    return reward

def compute_probability_without_flow(guidance_llm, context, explanation, true_action, possible_actions):
    """
    Tính reward (xác suất) cho toàn bộ explanation dựa trên Guidance LLM mà không dùng Flow Matching Model.
    
    Input:
        - guidance_llm (GuidanceLLM): LLM đã được load sẵn.
        - context (str): ngữ cảnh/quan sát.
        - explanation (str): giải thích đầy đủ (có thể gồm nhiều câu).
        - true_action (str): hành động đúng (Label), ví dụ: "positive" hoặc "negative".
        - possible_actions (list): danh sách các hành động có thể, ví dụ: ["negative", "positive"].
    
    Output:
        - probability (float): xác suất mà Guidance LLM gán cho hành động đúng (true_action).
    """
    # Truyền toàn bộ explanation vào LLM 
    prob_dist = guidance_llm.get_decision_distribution(
        user_input_text=context,
        explanation_sentences_so_far=explanation,
        all_possible_actions=possible_actions
    )

    # Lấy chỉ số của hành động đúng
    if true_action not in possible_actions:
        raise ValueError(f"True action '{true_action}' không nằm trong danh sách hành động: {possible_actions}")
    
    true_index = possible_actions.index(true_action)

    # Trả về xác suất của hành động đúng
    return prob_dist[true_index].item()


def split_response_into_sentences(response):
    """Tách explain thành các câu (dựa trên dấu chấm)."""
    if "Explanation:" in response:
        parts = response.split("Explanation:", 1)
        explain = parts[1].strip()

        response = explain.replace("\n", "")
    sentences = [s.strip() for s in response.split('.') if s.strip()]
    return sentences

def build_partial_responses(sentences):
    """Xây dựng các phần response: câu 1, câu 1+2, ..."""
    # if len(sentences) > 10:
    #     sentences = sentences[:10]
    partial_responses = []
    current_response = ""
    for s in sentences:
        current_response += (s + ".").strip()
        partial_responses.append(current_response)
    return partial_responses

def tuning_lm_with_rl_with_out_flow(args, guidance_llm=None, flow_model=None):
    # Khởi tạo script_args từ args
    script_args = args
    # print("reward_model_name:", script_args.reward_model_name)
    print("dataset_name:", script_args.datasets_dir)
    print("rl_base_model:", script_args.rl_base_model)

    # Cấu hình PPO
    config = PPOConfig(
        learning_rate=script_args.rl_learning_rate,
        batch_size=1,
        mini_batch_size=1,
        gradient_accumulation_steps=1,
        ppo_epochs=script_args.ppo_epochs,
        seed=script_args.seed,
    )

    # Tải tokenizer
    tokenizer = AutoTokenizer.from_pretrained(script_args.tokenizer_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Tạo dataset
    dataset = build_dataset(tokenizer, dataset_name=script_args.datasets_dir)
    print("Dataset created with", len(dataset), "samples")

    # Cấu hình LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    # Tải mô hình chính với value head
    model = AutoModelForCausalLMWithValueHead.from_pretrained(
        script_args.rl_base_model,
        torch_dtype=torch.bfloat16,
        peft_config=lora_config,
        use_gradient_checkpointing=True,  # Kích hoạt gradient checkpointing
        local_files_only=True
    )
    model.base_model_prefix = "model"
    model.generation_config = GenerationConfig.from_pretrained(
        script_args.rl_base_model,
        trust_remote_code=True
    )
    if not hasattr(model, "generation_config"):
        model.generation_config = GenerationConfig.from_pretrained(script_args.rl_base_model, trust_remote_code=True)
    print("Finetune model:", script_args.rl_base_model, type(model))

    # Khởi tạo Guidance LLM và FlowModel (giả sử đã có)
    if guidance_llm is None:
        raise SystemExit("Lỗi: guidance_llm chưa được truyền vào hàm tuning_lm_with_rl(). Vui lòng cung cấp mô hình hướng dẫn.")
    else:
        guidance_llm = guidance_llm

    if flow_model is None:
        raise SystemExit("Lỗi: flow_model chưa được truyền vào hàm tuning_lm_with_rl(). Vui lòng cung cấp mô hình Flow Matching đã huấn luyện.")
    else:
        flow_model = flow_model


    # Tạo optimizer (nếu dùng Adafactor)
    optimizer = None

    # Khởi tạo PPOTrainer
    ppo_trainer = PPOTrainer(
        config=config,
        model=model,
        tokenizer=tokenizer,
        dataset=dataset,
        data_collator=collator,
        optimizer=optimizer,
    )

    # Cấu hình tham số sinh văn bản
    generation_kwargs = {
        "top_k": 50,               # Thêm top_k để tránh mẫu quá ngẫu nhiên
        "top_p": 0.9,              # Giới hạn top_p để kiểm soát phân phối xác suất
        "temperature": 0.6,        # Giảm temperature để giảm độ nhiễu
        "do_sample": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "max_new_tokens": 150,
    }

    # Danh sách hành động/quyết định có thể
    possible_actions = ["Negative", "Positive"]  # Thay bằng danh sách thực tế


    # Vòng lặp huấn luyện PPO
    for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
        question_tensors = batch["input_ids"]
        true_actions = batch["label"]
        queries = batch["query"]

        # Sinh phản hồi (explanation)
        response_tensors = ppo_trainer.generate(
            question_tensors,
            return_prompt=False,
            **generation_kwargs,
        )
        batch["response"] = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

        # Tại đây tôi muốn thực PPO với với từng batch["query"]. Với mỗi response hãy tách ra thành nhiều dòng, mỗi dòng là thêm 1 câu của Response. VD( câu 1, câu 1 câu 2,....)
        # tính tổng điểm và huấn luyện PPO
        

        # Tính điểm thưởng từ Flow Matching Model (thay thế cho sentiment pipeline)
        for i in range(len(batch["query"])):
            response = batch["response"][i]
            target, explain = split_completion(response)
            query = queries[i]
            true_action = true_actions[i]
            question_tensor = question_tensors[i]

            # Tách explain thành các câu
            sentences = split_response_into_sentences(explain)
            if not sentences:
                continue
            # Xây dựng các phần explain: câu 1, câu 1+2, ...
            partial_explains = build_partial_responses(sentences)
            # Tokenize tất cả partial responses cùng lúc
            tokenized_responses = tokenizer(partial_explains, return_tensors="pt", truncation=True, max_length=256, padding=True).input_ids.to(DEVICE)

            # Sinh response từng phần (tokenize lại từng phần response) và Tính điểm cho từng phần
            pre_probability = 1/len(possible_actions)
            for i, response_tensor in enumerate(tokenized_responses):
                
                new_probability = compute_probability_without_flow(
                    guidance_llm,
                    query,
                    partial_explains[i],
                    true_action,
                    possible_actions
                )

                reward = new_probability - pre_probability
                pre_probability = new_probability

                # # Tokenize lại từng phần response
                # response_tensor = tokenizer(partial_response, return_tensors="pt", truncation=True, max_length=512).input_ids.squeeze(0)

                # Thực hiện bước PPO
                stats = ppo_trainer.step([question_tensor], [response_tensor], [torch.tensor(reward)])
                ppo_trainer.log_stats(stats, {"query": [query], "response": [partial_explains[i]]}, [reward])

                # Giải phóng bộ nhớ
                del response_tensor
                torch.cuda.empty_cache()
        # Lưu checkpoint định kỳ
        if script_args.save_freq and epoch and epoch % script_args.save_freq == 0:
            save_dir = os.path.join(script_args.output_dir, f"step_{epoch}")
            ppo_trainer.save_pretrained(save_dir)
            print(f"Saved checkpoint at: {save_dir}")

    # Lưu checkpoint cuối cùng
    final_save_dir = os.path.join(script_args.output_dir, "step_saved")
    ppo_trainer.save_pretrained(final_save_dir)
    print(f"Final checkpoint saved at: {final_save_dir}")

#======================================================================================================================================


# Khởi tạo lại GuidanceLLM (nếu chưa có)
guidance_llm = GuidanceLLM()


# Load lại state_dict
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

# Chạy hàm
tuning_lm_with_rl_with_out_flow(args, guidance_llm, guidance_llm)

# # Gộp adapter LoRA
from predict_module.merge_peft_adapter import merge_peft_adapter
merge_peft_adapter(
    model_name=os.path.join(args.output_dir, "step_saved"),
    output_name="./saved_models/SEPP_without_flow_model"
)


### TEST

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import pandas as pd
import re
import json

def split_completion(completion_text):
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)

        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            target = match.group(1).capitalize()
        else:
            target = 'Mixed'

        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        target = 'Mixed'
        explain = ""
        
    return target, explain

# Đường dẫn model và tokenizer
model_path = "./saved_models/SEPP_without_flow_model"
tokenizer_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)

# Load dataset từ CSV tesst/KLTN/Data/summarized/OpenAILLM_top1_stock_data_test.csv
data_path = "../Data/summarized/OpenAILLM_top1_stock_data_test.csv"
test_ds = pd.read_csv(data_path)

# Chuẩn bị danh sách lưu kết quả
data_result = []

# Prompt template (bạn cần chắc chắn rằng PREDICT_INSTRUCTION và PREDICT_EXAMPLES đã được định nghĩa trước)

# Lặp qua từng mẫu dữ liệu
i = 1
for i, sample in test_ds.iterrows():
    print(f"ĐANG THỰC HIỆN MẪU i = {i}")
    i += 1

    ticker = sample["ticker"]
    summary = sample["summary"]
    label = sample["target"]
    
    prompt = PREDICT_INSTRUCTION.format(
        examples=PREDICT_EXAMPLES,
        ticker=ticker,
        summary=summary
    )

    # print("\n--- Prompt ---\n", prompt)

    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        top_p=0.95,
        top_k=0,
        temperature=0.6,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode và in ra response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n--- Response ---\n", response)

    predict, explain = split_completion(response)

    data_policy.append({
        "user_input": prompt,
        "prediction_of_LLM": predict,
        "explain": explain,
        "Label": label
    })

# Lưu kết quả ra file CSV
df = pd.DataFrame(data_result)
df.to_csv("SEPP_without_flow_model_RESULTS_top1_stock.csv", index=False, encoding="utf-8-sig")

# Dọn dẹp bộ nhớ GPU
torch.cuda.empty_cache()
from sklearn.metrics import accuracy_score, matthews_corrcoef

# Giả sử df là DataFrame của bạn
y_pred = df["prediction_of_LLM"]
y_true = df["Label"]

# Tính accuracy
acc = accuracy_score(y_true, y_pred)

# Tính MCC
mcc = matthews_corrcoef(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"MCC: {mcc:.4f}")

## PPO ONLY

### Tải model mới về và lưu

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
# from trl import SFTTrainer
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict


def download_and_save_model(args):
      
    OUTPUT_DIR = args.output_path  # Thư mục lưu mô hình

    #==============================================================================================================================
    print(f"Đang tải mô hình từ: {args.model_path}")  # In đường dẫn mô hình

    # Step 3: Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        args.model_path,
        add_eos_token=True,  # Thêm token kết thúc
        local_files_only=args.offline if hasattr(args, 'offline') else False  # Chế độ offline
        )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Gán pad_token

    model = AutoModelForCausalLM.from_pretrained(
        args.model_path, 
        torch_dtype=torch.float16, 
        device_map="auto")

    
    model.save_pretrained(OUTPUT_DIR) 

# --- Tải và lưu mô hình ---

download_and_save_model(args)

Đang tải mô hình từ: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


### Huấn luyện PPO

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
from datasets import load_dataset, concatenate_datasets
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    GenerationConfig, DataCollatorWithPadding
)
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from peft import LoraConfig
import os

# # Thiết lập seed
# fix_seed = 100
# random.seed(fix_seed)
# torch.manual_seed(fix_seed)
# np.random.seed(fix_seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
POSSIBLE_ACTIONS = ["Negative", "Positive"]
# Cấu hình mô hình Flow Matching và Guidance LLM (giả sử đã có)
# Dưới đây là khởi tạo giả định, bạn sửa lại theo code thực tế của bạn
# from rectified_flow_model import RectifiedFlowModel, GuidanceLLM  # Giả sử đã import class này từ file khác

def build_dataset(tokenizer, dataset_name, input_min_text_length=2, input_max_text_length=8):
    """
    Tạo dataset cho huấn luyện PPO.
    Input:
        - tokenizer: Tokenizer để mã hóa văn bản.
        - dataset_name: Tên hoặc đường dẫn dataset.
        - input_min_text_length, input_max_text_length: giữ nguyên
    Output:
        - Dataset đã được xử lý với các cột query, input_ids, label.
    """
    ds = load_dataset(dataset_name, split="train")
    # repeat_factor = 8 // len(ds) + 1
    # ds = concatenate_datasets([ds] * repeat_factor)
    # ds = ds.select(range(8))
    original_columns = ds.column_names

    def preprocess_function(examples):
        new_examples = {
            "query": [],
            "input_ids": [],
            "label": [],
        }
        for context, label in zip(examples["user_input"], examples["Label"]):
            query = "Given the context: '" + context + "'. Please analyze reasoning for the agent decision based on the context."
            tokenized_question = tokenizer(
                query,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )
            new_examples["query"].append(query)
            new_examples["input_ids"].append(tokenized_question["input_ids"].squeeze(0))
            new_examples["label"].append(label)
        return new_examples

    ds = ds.map(
        preprocess_function,
        batched=True,
        num_proc=1,
        remove_columns=original_columns,
    )
    ds.set_format(type="torch")
    return ds

def collator(data):
    """
    Collator để xử lý batch dữ liệu.
    Input: list[dict] với các key: query, input_ids, label
    Output: dict với các key: query, input_ids, label
    """
    return_dict = {
        "query": [d["query"] for d in data],
        "input_ids": [d["input_ids"].to(DEVICE) for d in data],  # Giữ danh sách tensor 1D
        "label": [d["label"] for d in data],
    }
    return return_dict
    return dict((key, [d[key] for d in data]) for key in data[0])
    # return {
    #     "query": [d["query"] for d in data],
    #     "input_ids": [d["input_ids"] for d in data],
    #     "label": [d["label"] for d in data],
    # }

def compute_reward(flow_model, guidance_llm, context, explanation, true_action, possible_actions):
    """
    Tính reward cho toàn bộ explanation dựa trên Flow Matching Model.
    Input:
        - flow_model: RectifiedFlowModel đã huấn luyện
        - guidance_llm: GuidanceLLM
        - context (str): ngữ cảnh/quan sát
        - explanation (str): giải thích đầy đủ
        - true_action (str): hành động đúng (Label)
        - possible_actions (list): danh sách hành động/quyết định có thể
    Output:
        - reward (float): reward cho explanation hiện tại 
    """
    
    hidden_states = guidance_llm.get_last_layer_hidden_states(
        context, explanation, ", ".join(possible_actions)
    )
    z0 = torch.randn(1, len(possible_actions)).to(DEVICE)
    p_hat_dist = flow_model.solve_ode_generate_distribution(z0, hidden_states, num_steps=5)
    true_idx = possible_actions.index(true_action)
    reward = p_hat_dist[true_idx].item()

    del hidden_states, z0, p_hat_dist
    torch.cuda.empty_cache()
        
    return reward

def compute_probability_without_flow(guidance_llm, context, explanation, true_action, possible_actions):
    """
    Tính reward (xác suất) cho toàn bộ explanation dựa trên Guidance LLM mà không dùng Flow Matching Model.
    
    Input:
        - guidance_llm (GuidanceLLM): LLM đã được load sẵn.
        - context (str): ngữ cảnh/quan sát.
        - explanation (str): giải thích đầy đủ (có thể gồm nhiều câu).
        - true_action (str): hành động đúng (Label), ví dụ: "positive" hoặc "negative".
        - possible_actions (list): danh sách các hành động có thể, ví dụ: ["negative", "positive"].
    
    Output:
        - probability (float): xác suất mà Guidance LLM gán cho hành động đúng (true_action).
    """
    # Truyền toàn bộ explanation vào LLM 
    prob_dist = guidance_llm.get_decision_distribution(
        user_input_text=context,
        explanation_sentences_so_far=explanation,
        all_possible_actions=possible_actions
    )

    # Lấy chỉ số của hành động đúng
    if true_action not in possible_actions:
        raise ValueError(f"True action '{true_action}' không nằm trong danh sách hành động: {possible_actions}")
    
    true_index = possible_actions.index(true_action)

    # Trả về xác suất của hành động đúng
    return prob_dist[true_index].item()


def split_response_into_sentences(response):
    """Tách explain thành các câu (dựa trên dấu chấm)."""
    if "Explanation:" in response:
        parts = response.split("Explanation:", 1)
        explain = parts[1].strip()

        response = explain.replace("\n", "")
    sentences = [s.strip() for s in response.split('.') if s.strip()]
    return sentences

def build_partial_responses(sentences):
    """Xây dựng các phần response: câu 1, câu 1+2, ..."""
    # if len(sentences) > 10:
    #     sentences = sentences[:10]
    partial_responses = []
    current_response = ""
    for s in sentences:
        current_response += (s + ".").strip()
        partial_responses.append(current_response)
    return partial_responses

def tuning_lm_with_rl(args, guidance_llm=None, flow_model=None):
    # Khởi tạo script_args từ args
    script_args = args
    print("reward_model_name:", script_args.reward_model_name)
    print("dataset_name:", script_args.datasets_dir)
    print("rl_base_model:", script_args.rl_base_model)

    # Cấu hình PPO
    config = PPOConfig(
        learning_rate=script_args.rl_learning_rate,
        batch_size=1,
        mini_batch_size=1,
        gradient_accumulation_steps=1,
        ppo_epochs=script_args.ppo_epochs,
        seed=script_args.seed,
    )

    # Tải tokenizer
    tokenizer = AutoTokenizer.from_pretrained(script_args.tokenizer_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Tạo dataset
    dataset = build_dataset(tokenizer, dataset_name=script_args.datasets_dir)
    print("Dataset created with", len(dataset), "samples")

    # Cấu hình LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    # Tải mô hình chính với value head
    model = AutoModelForCausalLMWithValueHead.from_pretrained(
        script_args.rl_base_model,
        torch_dtype=torch.bfloat16,
        peft_config=lora_config,
        use_gradient_checkpointing=True,  # Kích hoạt gradient checkpointing
        local_files_only=True
    )
    model.base_model_prefix = "model"
    model.generation_config = GenerationConfig.from_pretrained(
        script_args.rl_base_model,
        trust_remote_code=True
    )
    if not hasattr(model, "generation_config"):
        model.generation_config = GenerationConfig.from_pretrained(script_args.rl_base_model, trust_remote_code=True)
    print("Finetune model:", script_args.rl_base_model, type(model))

    # Khởi tạo Guidance LLM và FlowModel (giả sử đã có)
    if guidance_llm is None:
        raise SystemExit("Lỗi: guidance_llm chưa được truyền vào hàm tuning_lm_with_rl(). Vui lòng cung cấp mô hình hướng dẫn.")
    else:
        guidance_llm = guidance_llm

    if flow_model is None:
        raise SystemExit("Lỗi: flow_model chưa được truyền vào hàm tuning_lm_with_rl(). Vui lòng cung cấp mô hình Flow Matching đã huấn luyện.")
    else:
        flow_model = flow_model


    # Tạo optimizer (nếu dùng Adafactor)
    optimizer = None

    # Khởi tạo PPOTrainer
    ppo_trainer = PPOTrainer(
        config=config,
        model=model,
        tokenizer=tokenizer,
        dataset=dataset,
        data_collator=collator,
        optimizer=optimizer,
    )

    # Cấu hình tham số sinh văn bản
    generation_kwargs = {
        "top_k": 50,               # Thêm top_k để tránh mẫu quá ngẫu nhiên
        "top_p": 0.9,              # Giới hạn top_p để kiểm soát phân phối xác suất
        "temperature": 0.6,        # Giảm temperature để giảm độ nhiễu
        "do_sample": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "max_new_tokens": 150,
    }

    # Danh sách hành động/quyết định có thể
    possible_actions = ["Negative", "Positive"]  # Thay bằng danh sách thực tế


    # Vòng lặp huấn luyện PPO
    for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
        question_tensors = batch["input_ids"]
        true_actions = batch["label"]
        queries = batch["query"]

        # Sinh phản hồi (explanation)
        response_tensors = ppo_trainer.generate(
            question_tensors,
            return_prompt=False,
            **generation_kwargs,
        )
        batch["response"] = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

        # Tại đây tôi muốn thực PPO với với từng batch["query"]. Với mỗi response hãy tách ra thành nhiều dòng, mỗi dòng là thêm 1 câu của Response. VD( câu 1, câu 1 câu 2,....)
        # tính tổng điểm và huấn luyện PPO
        

        # Tính điểm thưởng từ Flow Matching Model (thay thế cho sentiment pipeline)
        for i in range(len(batch["query"])):
            response = batch["response"][i]
            target, explain = split_completion(response)
            query = queries[i]
            true_action = true_actions[i]
            question_tensor = question_tensors[i]

            # Tách explain thành các câu
            sentences = split_response_into_sentences(explain)
            if not sentences:
                continue
            # Xây dựng các phần explain: câu 1, câu 1+2, ...
            partial_explains = build_partial_responses(sentences)
            # Tokenize tất cả partial responses cùng lúc
            tokenized_responses = tokenizer(partial_explains, return_tensors="pt", truncation=True, max_length=256, padding=True).input_ids.to(DEVICE)

            # Sinh response từng phần (tokenize lại từng phần response) và Tính điểm cho từng phần
            pre_probability = 1/len(possible_actions)
            for i, response_tensor in enumerate(tokenized_responses):
                
                new_probability = compute_reward(
                    flow_model,
                    guidance_llm,
                    query,
                    partial_explains[i],
                    true_action,
                    possible_actions
                )

                reward = new_probability - pre_probability
                pre_probability = new_probability

                # # Tokenize lại từng phần response
                # response_tensor = tokenizer(partial_response, return_tensors="pt", truncation=True, max_length=512).input_ids.squeeze(0)

                # Thực hiện bước PPO
                stats = ppo_trainer.step([question_tensor], [response_tensor], [torch.tensor(reward)])
                ppo_trainer.log_stats(stats, {"query": [query], "response": [partial_explains[i]]}, [reward])

                # Giải phóng bộ nhớ
                del response_tensor
                torch.cuda.empty_cache()
        # Lưu checkpoint định kỳ
        if script_args.save_freq and epoch and epoch % script_args.save_freq == 0:
            save_dir = os.path.join(script_args.output_dir, f"step_{epoch}")
            ppo_trainer.save_pretrained(save_dir)
            print(f"Saved checkpoint at: {save_dir}")

    # Lưu checkpoint cuối cùng
    final_save_dir = os.path.join(script_args.output_dir, "step_saved")
    ppo_trainer.save_pretrained(final_save_dir)
    print(f"Final checkpoint saved at: {final_save_dir}")


# Khởi tạo lại GuidanceLLM (nếu chưa có)
guidance_llm = GuidanceLLM()

# Khởi tạo cấu trúc flow_model
flow_model_loaded = RectifiedFlowModel(
    guidance_llm_instance=guidance_llm,
    num_actions=len(POSSIBLE_ACTIONS),
    flow_embed_dim=128,
    projector_hidden_dim=128,
    num_attention_heads=2
).to(DEVICE).to(torch.bfloat16)

# Load lại state_dict
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

# Đường dẫn lưu model
flow_model_path = os.path.join(save_dir, "flow_model.pt")
flow_model_loaded.load_state_dict(torch.load(flow_model_path, map_location=DEVICE))
flow_model_loaded.eval()

# Chạy hàm
tuning_lm_with_rl(args, guidance_llm, flow_model_loaded)

# # Gộp adapter LoRA
from predict_module.merge_peft_adapter import merge_peft_adapter
merge_peft_adapter(
    model_name=os.path.join(args.output_dir, "step_saved"),
    output_name="./saved_models/SEPP_PPO_only_model"
)


### Test

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import pandas as pd
import re
import json

def split_completion(completion_text):
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)

        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            target = match.group(1).capitalize()
        else:
            target = 'Mixed'

        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        target = 'Mixed'
        explain = ""
        
    return target, explain

# Đường dẫn model và tokenizer
model_path = "./saved_models/SEPP_PPO_only_model"
tokenizer_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)

# Load dataset từ CSV tesst/KLTN/Data/summarized/OpenAILLM_top1_stock_data_test.csv
data_path = "../Data/summarized/OpenAILLM_top1_stock_data_test.csv"
test_ds = pd.read_csv(data_path)

# Chuẩn bị danh sách lưu kết quả
data_result = []

# Prompt template (bạn cần chắc chắn rằng PREDICT_INSTRUCTION và PREDICT_EXAMPLES đã được định nghĩa trước)

# Lặp qua từng mẫu dữ liệu
i = 1
for i, sample in test_ds.iterrows():
    print(f"ĐANG THỰC HIỆN MẪU i = {i}")
    i += 1

    ticker = sample["ticker"]
    summary = sample["summary"]
    label = sample["target"]

    prompt = PREDICT_INSTRUCTION.format(
        examples=PREDICT_EXAMPLES,
        ticker=ticker,
        summary=summary
    )

    # print("\n--- Prompt ---\n", prompt)

    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        top_p=0.95,
        top_k=0,
        temperature=0.6,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode và in ra response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n--- Response ---\n", response)

    predict, explain = split_completion(response)

    data_result.append({
        "user_input": prompt,
        "prediction_of_LLM": predict,
        "explain": explain,
        "Label": label
    })

# Lưu kết quả ra file CSV
df = pd.DataFrame(data_result)
df.to_csv("SEPP_PPO_only_model_RESULTS_top1_stock.csv", index=False, encoding="utf-8-sig")

# Dọn dẹp bộ nhớ GPU
torch.cuda.empty_cache()
from sklearn.metrics import accuracy_score, matthews_corrcoef

# Giả sử df là DataFrame của bạn
y_pred = df["prediction_of_LLM"]
y_true = df["Label"]

# Tính accuracy
acc = accuracy_score(y_true, y_pred)

# Tính MCC
mcc = matthews_corrcoef(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"MCC: {mcc:.4f}")